In [2]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import os

In [3]:
CLASSES = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']
MODEL_PATH = 'garbage_model.pth'
DEFAULT_IMAGE = 'test_foto.jpg'

In [4]:
def load_checkpoint(filepath):
    model = models.resnet50(weights=None)
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, len(CLASSES))
    model.load_state_dict(torch.load(filepath, map_location=torch.device('cpu')))
    model.eval()
    return model

In [5]:
def predict_image(image_path, model):
    transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
 
    image = Image.open(image_path).convert('RGB')
    image = transform(image).unsqueeze(0)
 
    with torch.no_grad():
        output = model(image)
        probabilities = torch.nn.functional.softmax(output[0], dim=0)
 
    top3_prob, top3_idx = torch.topk(probabilities, 3)
    results = [(CLASSES[idx], prob.item()) for idx, prob in zip(top3_idx, top3_prob)]
    return results
 

In [20]:
if __name__ == '__main__':
    if not os.path.exists(MODEL_PATH):
        print(f"Błąd: Nie znaleziono pliku '{MODEL_PATH}'. Najpierw uruchom trening!")
        exit(1)
 
    print(f"Podaj ścieżkę do zdjęcia (Enter = '{DEFAULT_IMAGE}'): ", end='')
    image_path = input().strip() or DEFAULT_IMAGE
 
    if not os.path.exists(image_path):
        print(f"Błąd: Nie znaleziono pliku '{image_path}'.")
        exit(1)
 
    print(f"\nAnalizuję: {image_path}...")
    net = load_checkpoint(MODEL_PATH)
    results = predict_image(image_path, net)
 
    print("\n--- Wyniki analizy ---")
    for i, (label, score) in enumerate(results, 1):
        bar = '█' * int(score * 20)
        print(f"{i}. {label:<12} {score*100:5.1f}%  {bar}")

Podaj ścieżkę do zdjęcia (Enter = 'test_foto.jpg'): 

 wino.jpg



Analizuję: wino.jpg...

--- Wyniki analizy ---
1. glass        100.0%  ███████████████████
2. plastic        0.0%  
3. cardboard      0.0%  
